# Prompt Engineering Lab: TechFlow Solutions Chatbot

TechFlow Solutions' customer service chatbot gives inconsistent answers on three tasks: sentiment analysis, product description generation and data extraction. This notebook diagnoses the original prompts by running them 5, 10 and 15 times, then improves them in two iterations: v2 (clarity, format, constraints) and v3 (few-shot examples, Chain-of-Thought).


## Part 1: Create Initial Prompts

### Step 1: Setup Your Environment

In [1]:
# 1.1 Install the required libraries (only needed once)

%pip install openai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
# 1.2 Import libraries and connect to OpenAI (setup cell from the course notebook)

import os
import re
import json
import time
from collections import Counter

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

# Load the API key from the .env file, so it never appears in the notebook
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# One model for the whole lab, so results stay comparable
MODEL = "gpt-4o-mini"

print("✅ Setup complete! OpenAI client initialized.")

✅ Setup complete! OpenAI client initialized.


In [2]:
# 1.3 Helper functions (course notebook cell)

def call_openai(prompt, system_message="You are a helpful assistant.", temperature=0.7, max_tokens=None, seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }

        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed

        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def display_response(title, response, params=None):
    """Display a response with formatting"""
    print(f"\n{'='*60}")
    print(f"📝 {title}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")

def count_tokens_approx(text):
    """Approximate token count (roughly 4 characters per token)"""
    return len(text) // 4

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


In [3]:
# 1.4 Helper for this lab: run the same prompt several times and collect all answers

def run_multiple(prompt, n_runs=5, system_message="You are a helpful assistant.", temperature=0.7):
    responses = []
    for i in range(n_runs):
        response = call_openai(prompt, system_message=system_message, temperature=temperature)
        responses.append(response)
        time.sleep(0.5)  # Small delay to avoid rate limiting
    return responses


# Quick test 1: one call, shown with the course display helper
response = call_openai("Say 'Hello, TechFlow!' and nothing else.")
display_response("Quick test: one call", response, {"model": MODEL, "temperature": 0.7})

# Quick test 2: for Lab, same prompt 3 times
test_responses = run_multiple("Write a haiku about AI", n_runs=3)
for i, test_response in enumerate(test_responses, start=1):
    print(f"🔄 Run #{i}:\n{test_response}\n")
print(f"📊 Total unique responses: {len(set(test_responses))} out of 3")
print("✅ Setup checkpoint passed: all helpers work")


📝 Quick test: one call
Parameters: {'model': 'gpt-4o-mini', 'temperature': 0.7}
Hello, TechFlow!

🔄 Run #1:
Silent circuits hum,  
Thoughts of silicon arise,  
Dreams in code take flight.

🔄 Run #2:
Lines of code entwined,  
Minds of silicon awaken—  
Dreams in circuits born.  

🔄 Run #3:
Silent thoughts emerge,  
Lines of code weave dreams and truth,  
Mind of metal learns.

📊 Total unique responses: 3 out of 3
✅ Setup checkpoint passed: all helpers work


### Step 2: Create Initial Prompts

In [4]:
# 2.1 Task 1: Sentiment Analysis

# Initial simple prompt
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# Test it once
result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

Sentiment Analysis Result:
This customer message can be classified as **positive feedback** or **customer satisfaction**.


In [5]:
# 2.2 Task 2: Product Description Generation

# Initial simple prompt
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

# Test it once
result = call_openai(product_prompt_v1)
print("Product Description Result:")
print(result)

Product Description Result:
**Product Description: Wireless Freedom Mouse - $29.99**

Elevate your productivity and comfort with our Wireless Freedom Mouse, designed for the modern user who values both style and functionality. Priced at an affordable $29.99, this sleek and ergonomic mouse is the perfect companion for your laptop or desktop, whether you're working from home, in the office, or on the go.

**Key Features:**

- **Wireless Convenience:** Say goodbye to tangled cords! Our advanced 2.4GHz wireless technology provides a reliable and fast connection, giving you the freedom to move without restrictions. Enjoy a seamless range of up to 33 feet, allowing you to work comfortably from anywhere in the room.

- **Ergonomic Design:** Crafted with user comfort in mind, the Wireless Freedom Mouse features a contoured shape that fits snugly in your hand, reducing strain during long hours of use. The textured grip ensures precise control, making it perfect for both casual browsing and inte

In [6]:
# 2.3 Task 3: Data Extraction

# Initial simple prompt
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Data Extraction Result:
Here is the extracted information from the customer feedback:

- **Item Number:** 12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged


## Part 2: Diagnosing Failures - Systematic Testing
### Step 3: Run Prompts 5 Times
Each answer is first reduced to what should stay the same, then compared:
- **Sentiment:** the label itself (capitals, punctuation and spaces ignored)
- **Product description:** the format (list lines + length band), because the wording should change every time
- **Data extraction:** the extracted data if the answer is valid JSON, otherwise the raw text

**Consistency %** = share of runs that give the most common answer. **Unique answers** = how many different answers appeared.

In [7]:
# 3.1 Comparison rules: what must stay the same for two answers to count as "the same"

# Sentiment: the label itself, ignoring capitals, punctuation and spaces
def sentiment_key(text):
    return re.sub(r"[^a-z ]", "", " ".join(text.lower().split())).strip()

# Product description: the FORMAT, because the wording should change every time
# Format = lines starting with a list or Markdown marker (-, •, *) + length band
# The bands are wide on purpose: an answer that follows the v2/v3 template
# (about 60-100 words) always lands in the same band
def product_key(text):
    list_lines = sum(1 for line in text.splitlines() if line.strip().startswith(("-", "•", "*")))
    words = len(text.split())
    if words < 50:
        length = "short (<50 words)"
    elif words <= 120:
        length = "medium (50-120 words)"
    else:
        length = "long (>120 words)"
    return f"{list_lines} list lines | {length}"

# Data extraction: the extracted data if the answer is valid JSON, otherwise the raw text
def extraction_key(text):
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.dumps(json.loads(cleaned), sort_keys=True)
    except json.JSONDecodeError:
        return "NOT JSON: " + " ".join(text.lower().split())

print("✅ Comparison rules ready")

✅ Comparison rules ready


In [8]:
# 3.2 Test helper: run a prompt n times, print every answer, measure consistency, record the result

all_results = []  # one row per test, used for every table in this notebook

def test_prompt(task, version, prompt, key_function, n_runs, system_message="You are a helpful assistant."):
    responses = run_multiple(prompt, n_runs, system_message)

    # Consistency = share of runs that give the most common answer
    keys = [key_function(r) for r in responses]
    top_answer, top_count = Counter(keys).most_common(1)[0]
    consistency = top_count / n_runs * 100
    unique = len(set(keys))

    print(f"\n{'='*60}")
    print(f"📝 {task} | {version} | {n_runs} runs")
    print(f"{'='*60}")
    for i, r in enumerate(responses, start=1):
        print(f"🔄 Run #{i}: {' '.join(r.split())[:150]}")
    print(f"\n📊 Consistency: {consistency:.0f}% | Unique answers: {unique} out of {n_runs}")
    print(f"Most common answer: {top_answer[:150]}")

    all_results.append({"Task": task, "Version": version, "Runs": n_runs,
                        "Consistency %": round(consistency), "Unique answers": unique})
    return responses

print("✅ Test helper ready")

✅ Test helper ready


In [9]:
# 3.3 Run each v1 prompt 5 times and record all responses

sentiment_v1_5 = test_prompt("Sentiment", "v1", sentiment_prompt_v1, sentiment_key, 5)
product_v1_5 = test_prompt("Product", "v1", product_prompt_v1, product_key, 5)
extraction_v1_5 = test_prompt("Extraction", "v1", extraction_prompt_v1, extraction_key, 5)


📝 Sentiment | v1 | 5 runs
🔄 Run #1: This customer message can be classified as **positive feedback** or **satisfaction**. The customer expresses love for the product and indicates that i
🔄 Run #2: This customer message can be classified as positive feedback or a positive review.
🔄 Run #3: The customer message can be classified as positive feedback.
🔄 Run #4: The customer message can be classified as positive feedback or a positive review.
🔄 Run #5: The customer message can be classified as positive feedback or a positive review.

📊 Consistency: 40% | Unique answers: 4 out of 5
Most common answer: the customer message can be classified as positive feedback or a positive review

📝 Product | v1 | 5 runs
🔄 Run #1: **Product Description: Wireless Precision Mouse - $29.99** Elevate your productivity with our Wireless Precision Mouse, designed for seamless navigati
🔄 Run #2: **Product Name: SwiftConnect Wireless Mouse** **Price: $29.99** **Description:** Elevate your workspace with the Swift

In [10]:
# 3.4 Consistency of the three prompts after 5 runs

pd.DataFrame(all_results)

,Task,Version,Runs,Consistency %,Unique answers
0,Sentiment,v1,5,40,4
1,Product,v1,5,40,4
2,Extraction,v1,5,20,5


### Step 3: Observations after 5 runs

| Task | Consistency | Unique answers |
|---|---|---|
| Sentiment | 40% | 4 of 5 |
| Product description | 40% | 4 of 5 |
| Data extraction | 20% | 5 of 5 |

**What varied in each task**
- **Sentiment:** always a full sentence ("The customer message can be classified as..."), never a one-word label. 4 of 5 answers offered two labels ("positive feedback or a positive review").
- **Product description:** Markdown texts with bold headings and a different number of list lines. Two layouts appeared (one heading line, or separate "Product Name" and "Price" lines), with four different invented product names in five runs.
- **Data extraction:** never JSON. Every answer was an intro sentence plus a bullet list, and the field names ("Delivery" vs "Delivery Speed", "Packaging Condition" vs "Issue"), the number format ("#12345" vs "12345") and the bold formatting changed in every run.

**Answers to the Step 3 questions**
- **Consistent in format?** No. None of the three prompts kept a stable format, and extraction gave a different layout in every run.
- **Consistent in content?** Yes for the facts. The inconsistency is in how the answer is written, not in what it says. Product descriptions are the exception: the product name and specs are invented each time.
- **Types of variation:** a sentence instead of a label, one or two labels, added explanations, Markdown headings and bold text, invented product names, changing field names and number formats.

### Step 4: Run Prompts 10 Times

In [11]:
# 4.1 Run each v1 prompt 10 times

sentiment_v1_10 = test_prompt("Sentiment", "v1", sentiment_prompt_v1, sentiment_key, 10)
product_v1_10 = test_prompt("Product", "v1", product_prompt_v1, product_key, 10)
extraction_v1_10 = test_prompt("Extraction", "v1", extraction_prompt_v1, extraction_key, 10)


📝 Sentiment | v1 | 10 runs
🔄 Run #1: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
🔄 Run #2: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
🔄 Run #3: This customer message can be classified as positive feedback.
🔄 Run #4: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
🔄 Run #5: The customer message can be classified as "Positive Feedback" or "Satisfaction."
🔄 Run #6: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
🔄 Run #7: This customer message can be classified as positive feedback.
🔄 Run #8: This customer message can be classified as **positive feedback** or **customer satisfaction**.
🔄 Run #9: The customer message can be classified as **positive feedback** or **satisfaction**.
🔄 Run #10: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

📊 Consistency: 40% | U

In [13]:
# 4.2 Compare the 10-run test with the 5-run test
# (pivot_table with "last" keeps working if a test cell was run twice: it shows the latest run)

v1_tests = pd.DataFrame(all_results).query("Version == 'v1'")
v1_tests.pivot_table(index="Task", columns="Runs",
                     values=["Consistency %", "Unique answers"], aggfunc="last")

Consistency %     Unique answers   
Runs                  5   10             5  10
Task                                          
Extraction            20  20              5  9
Product               40  50              4  4
Sentiment             40  40              4  4

### Step 4: 10 runs vs. 5 runs

| Task | Consistency (5 → 10 runs) | Unique answers (5 → 10 runs) |
|---|---|---|
| Sentiment | 40% → 40% | 4 → 4 |
| Product description | 40% → 50% | 4 → 4 |
| Data extraction | 20% → 20% | 5 → 9 |

**Did consistency drop?**
No, it stayed low: 40% → 40% for sentiment, 20% → 20% for extraction and 40% → 50% for product descriptions. 

**New failure patterns at 10 runs**
- **Sentiment:** labels in quotation marks (3 runs) and in Title Case ("Positive Feedback", 6 runs), and the second label switched from "a positive review" to "customer satisfaction". The comparison rule ignores capitals and punctuation, so the real variety is larger than the score shows.
- **Product description:** titles with nested colons, for example "Wireless Freedom: Sleek & Ergonomic Wireless Mouse" (2 runs).
- **Data extraction:** new wording ("Here's" instead of "Here is") and a new field name ("Delivery Experience").

**Takeaway:** 5 runs understate the variety. Each run is worth 20 points, so a 5-run score is a coarse estimate, and answer types that were simply not sampled yet only show up with more runs.

### Step 5: Run Prompts 15 Times and Create Failure Analysis

In [14]:
# 5.1 Run each v1 prompt 15 times

sentiment_v1_15 = test_prompt("Sentiment", "v1", sentiment_prompt_v1, sentiment_key, 15)
product_v1_15 = test_prompt("Product", "v1", product_prompt_v1, product_key, 15)
extraction_v1_15 = test_prompt("Extraction", "v1", extraction_prompt_v1, extraction_key, 15)


📝 Sentiment | v1 | 15 runs
🔄 Run #1: This customer message can be classified as **positive feedback** or **customer satisfaction**.
🔄 Run #2: The customer message can be classified as positive feedback.
🔄 Run #3: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
🔄 Run #4: The customer message can be classified as positive feedback or a positive review.
🔄 Run #5: This customer message can be classified as **positive feedback** or **customer satisfaction**.
🔄 Run #6: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
🔄 Run #7: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
🔄 Run #8: This customer message can be classified as "Positive Feedback" or "Customer Praise."
🔄 Run #9: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
🔄 Run #10: The customer message can be classified as **Positive Feedback**.
🔄 Run #11: The 

In [15]:
# 5.2 Failure analysis table: v1 across 5, 10 and 15 runs

v1_tests = pd.DataFrame(all_results).query("Version == 'v1'")
failure_table = v1_tests.pivot_table(index="Task", columns="Runs",
                                     values=["Consistency %", "Unique answers"], aggfunc="last")
failure_table

Consistency %         Unique answers      
Runs                  5   10  15             5  10 15
Task                                                 
Extraction            20  20  47              5  9  8
Product               40  50  40              4  4  5
Sentiment             40  40  40              4  4  7

In [16]:
# 5.3 Every answer type in the 15-run test and how often it appeared (evidence for the failure patterns)

for task, responses, key_function in [("Sentiment", sentiment_v1_15, sentiment_key),
                                      ("Product", product_v1_15, product_key),
                                      ("Extraction", extraction_v1_15, extraction_key)]:
    print(f"\n📝 {task}:")
    for answer, count in Counter(key_function(r) for r in responses).most_common():
        print(f"  {count}x  {answer[:100]}")


📝 Sentiment:
  6x  the customer message can be classified as positive feedback or customer satisfaction
  3x  this customer message can be classified as positive feedback or customer satisfaction
  2x  the customer message can be classified as positive feedback
  1x  the customer message can be classified as positive feedback or a positive review
  1x  this customer message can be classified as positive feedback or customer praise
  1x  the customer message can be classified as positive feedback or satisfaction the customer expresses e
  1x  this customer message can be classified as positive feedback or a positive review

📝 Product:
  6x  8 list lines | long (>120 words)
  5x  9 list lines | long (>120 words)
  2x  10 list lines | long (>120 words)
  1x  12 list lines | long (>120 words)
  1x  16 list lines | long (>120 words)

📝 Extraction:
  7x  NOT JSON: here is the extracted information from the customer feedback: - **order number**: #12345 -
  2x  NOT JSON: here is the extracted

### Step 5: Failure analysis (v1, 15 runs)

| Task | Consistency (5 / 10 / 15 runs) | Unique answers (5 / 10 / 15 runs) |
|---|---|---|
| Sentiment | 40% / 40% / 40% | 4 / 4 / 7 |
| Product description | 40% / 50% / 40% | 4 / 4 / 5 |
| Data extraction | 20% / 20% / 47% | 5 / 9 / 8 |

**sentiment** (answer types in cell 5.3)
- Never a one-word label: all 15 answers were sentences ("The customer message can be classified as...").
- 13 of 15 answers offered two labels: "positive feedback or customer satisfaction", "or a positive review", "or customer praise".
- One answer added an explanation

**product description**
- Always long (more than 120 words) Markdown, with 8 to 16 list and heading lines.
- Two layouts: one heading line  or separate "Product Name" and "Price" lines; the price was in the heading in 8 of 15 runs.
- Five different invented product names

**data extraction**
- Never JSON (0 of 15): always an intro sentence plus a bullet list.
- "#12345" in 13 runs and "12345" in 2; bold placement and the other field names changed between runs.

**Root cause:** 
the v1 prompts do not say what format the answer must have, so the model chooses a different one each run. 
For sentiment and extraction the facts were stable (always positive, always the same order data); only the format varied. 
Product descriptions also add a content risk: the model invents names and specs that were never given.

**Why extraction jumped to 47% at 15 runs:** the small samples can mislead in both directions. That is why the versions are compared at 15 runs.

## Part 3: Iteration 1 - Rewriting Simple Prompts

### Step 6: Improve Sentiment Analysis Prompt
- **Role prompting** in the system message, written in the third person ("The assistant is...")
- **A clearer instruction** and an **explicit output format**: positive, negative or neutral
- **A single-word constraint**, placed at the end of the prompt (recency bias: the model follows the last instruction best)

In [17]:
# 6.1 Version 2: role in the system message, clear instruction, explicit format, one-word rule at the end

sentiment_system = "The assistant is a sentiment classifier for TechFlow Solutions' customer service team."

sentiment_prompt_v2 = """
Classify the sentiment of this customer message.

Customer message: "I love this product! It's exactly what I needed."

Answer with exactly one word: positive, negative or neutral. Use lowercase letters and no punctuation.
"""

display_response("Sentiment prompt v2", sentiment_prompt_v2,
                 {"system_message": sentiment_system,
                  "approx_tokens": count_tokens_approx(sentiment_prompt_v2)})


📝 Sentiment prompt v2
Parameters: {'system_message': "The assistant is a sentiment classifier for TechFlow Solutions' customer service team.", 'approx_tokens': 56}

Classify the sentiment of this customer message.

Customer message: "I love this product! It's exactly what I needed."

Answer with exactly one word: positive, negative or neutral. Use lowercase letters and no punctuation.




In [18]:
# 6.2 Test version 2 fifteen times

sentiment_v2_15 = test_prompt("Sentiment", "v2", sentiment_prompt_v2, sentiment_key, 15, sentiment_system)


📝 Sentiment | v2 | 15 runs
🔄 Run #1: positive
🔄 Run #2: positive
🔄 Run #3: positive
🔄 Run #4: positive
🔄 Run #5: positive
🔄 Run #6: positive
🔄 Run #7: positive
🔄 Run #8: positive
🔄 Run #9: positive
🔄 Run #10: positive
🔄 Run #11: positive
🔄 Run #12: positive
🔄 Run #13: positive
🔄 Run #14: positive
🔄 Run #15: positive

📊 Consistency: 100% | Unique answers: 1 out of 15
Most common answer: positive


### Step 7: Improve Product Description Prompt
Version 2 adds a role, 
**style guidelines** (tone, no emojis, no headings, no bold text), 
**length constraints** (title max 8 words, paragraph 40-60 words) and a 
**fixed structure**, placed at the end. The wording will still vary, which is wanted; the format should not.

In [19]:
# 7.1 Version 2: role, style guidelines, length limits and a fixed structure at the end

product_system = "The assistant is a copywriter for TechFlow Solutions' online electronics shop."

product_prompt_v2 = """
Write a product description for a wireless mouse that costs $29.99.

Style: friendly and professional. No emojis, no headings, no bold text.

Use exactly this structure:
Title: <product name and one key benefit, max 8 words>
<one paragraph of 40-60 words that mentions the price once>
- <feature 1>
- <feature 2>
- <feature 3>
"""

display_response("Product prompt v2", product_prompt_v2,
                 {"system_message": product_system,
                  "approx_tokens": count_tokens_approx(product_prompt_v2)})


📝 Product prompt v2
Parameters: {'system_message': "The assistant is a copywriter for TechFlow Solutions' online electronics shop.", 'approx_tokens': 82}

Write a product description for a wireless mouse that costs $29.99.

Style: friendly and professional. No emojis, no headings, no bold text.

Use exactly this structure:
Title: <product name and one key benefit, max 8 words>
<one paragraph of 40-60 words that mentions the price once>
- <feature 1>
- <feature 2>
- <feature 3>




In [20]:
# 7.2 Test version 2 fifteen times

product_v2_15 = test_prompt("Product", "v2", product_prompt_v2, product_key, 15, product_system)


📝 Product | v2 | 15 runs
🔄 Run #1: Title: SmoothGlide Wireless Mouse for Effortless Navigation Experience seamless control with the SmoothGlide Wireless Mouse, designed to enhance your 
🔄 Run #2: Title: Precision Wireless Mouse for Seamless Navigation Experience effortless control with our Precision Wireless Mouse, designed to enhance your prod
🔄 Run #3: Title: SwiftClick Wireless Mouse for Effortless Navigation Experience seamless productivity with the SwiftClick Wireless Mouse, priced at just $29.99.
🔄 Run #4: Title: SwiftConnect Wireless Mouse for Effortless Navigation Experience seamless control with the SwiftConnect Wireless Mouse, designed for comfort an
🔄 Run #5: Title: Sleek Wireless Mouse for Effortless Precision Discover seamless navigation with our wireless mouse, designed for both comfort and efficiency. P
🔄 Run #6: Title: SwiftClick Wireless Mouse for Effortless Navigation Experience a new level of productivity with the SwiftClick Wireless Mouse, designed for com
🔄 Run #7

### Step 8: Improve Data Extraction Prompt
Version 2 adds a role, a **structured output format** (JSON only), the **specific fields** to extract and a **clear format rule** at the end. The field values are still free text, so some variation (and possibly JSON parsing issues) is expected; Step 10 deals with it.

In [21]:
# 8.1 Version 2: role, specific fields, JSON-only output format at the end

extraction_system = "The assistant extracts order data from TechFlow Solutions' customer feedback."

extraction_prompt_v2 = """
Extract the order information from this customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

Return only a JSON object with exactly these fields:
"order_number", "order_date", "delivery_speed", "packaging_condition"
Do not write any text before or after the JSON.
"""

display_response("Extraction prompt v2", extraction_prompt_v2,
                 {"system_message": extraction_system,
                  "approx_tokens": count_tokens_approx(extraction_prompt_v2)})


📝 Extraction prompt v2
Parameters: {'system_message': "The assistant extracts order data from TechFlow Solutions' customer feedback.", 'approx_tokens': 81}

Extract the order information from this customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

Return only a JSON object with exactly these fields:
"order_number", "order_date", "delivery_speed", "packaging_condition"
Do not write any text before or after the JSON.




In [22]:
# 8.2 Test version 2 fifteen times

extraction_v2_15 = test_prompt("Extraction", "v2", extraction_prompt_v2, extraction_key, 15, extraction_system)


📝 Extraction | v2 | 15 runs
🔄 Run #1: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #2: { "order_number": "12345", "order_date": "2023-03-15", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #3: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #4: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #5: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #6: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #7: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "damaged" }
🔄 Run #8: { "order_number": "12345", "order_date": "March 15th", "delivery_speed": "fast"

In [23]:
# 8.3 Checkpoint: version 2 vs version 1 (15 runs)

tests_15 = pd.DataFrame(all_results).query("Runs == 15")
tests_15.pivot_table(index="Task", columns="Version", values="Consistency %", aggfunc="last")

Version,v1,v2
Task,,
Extraction,47,87
Product,40,100
Sentiment,40,100


## Part 4: Iteration 2 - Adding Structure & Constraints
### Step 9: Add Few-Shot Examples to Sentiment Analysis


In [24]:
# 9.1 Version 3: the v2 rules plus three examples in the exact answer format
# {message} is a placeholder, so Step 12 can reuse the prompt with new messages

sentiment_prompt_v3 = """
Classify the sentiment of the customer message as positive, negative or neutral.
Answer with exactly one lowercase word, like in the examples.

Customer message: "The app crashes every time I open it. Very disappointed."
Sentiment: negative

Customer message: "My package arrived on Tuesday."
Sentiment: neutral

Customer message: "Great service, the team solved my problem in minutes!"
Sentiment: positive

Customer message: "{message}"
Sentiment:"""

sentiment_prompt_v3_test = sentiment_prompt_v3.replace(
    "{message}", "I love this product! It's exactly what I needed.")
display_response("Sentiment prompt v3 (few-shot)", sentiment_prompt_v3_test,
                 {"system_message": sentiment_system,
                  "approx_tokens": count_tokens_approx(sentiment_prompt_v3_test)})


📝 Sentiment prompt v3 (few-shot)
Parameters: {'system_message': "The assistant is a sentiment classifier for TechFlow Solutions' customer service team.", 'approx_tokens': 122}

Classify the sentiment of the customer message as positive, negative or neutral.
Answer with exactly one lowercase word, like in the examples.

Customer message: "The app crashes every time I open it. Very disappointed."
Sentiment: negative

Customer message: "My package arrived on Tuesday."
Sentiment: neutral

Customer message: "Great service, the team solved my problem in minutes!"
Sentiment: positive

Customer message: "I love this product! It's exactly what I needed."
Sentiment:



In [25]:
# 9.2 Test the few-shot prompt fifteen times

sentiment_v3_15 = test_prompt("Sentiment", "v3", sentiment_prompt_v3_test, sentiment_key, 15, sentiment_system)


📝 Sentiment | v3 | 15 runs
🔄 Run #1: positive
🔄 Run #2: positive
🔄 Run #3: positive
🔄 Run #4: positive
🔄 Run #5: positive
🔄 Run #6: positive
🔄 Run #7: positive
🔄 Run #8: positive
🔄 Run #9: positive
🔄 Run #10: positive
🔄 Run #11: positive
🔄 Run #12: positive
🔄 Run #13: positive
🔄 Run #14: positive
🔄 Run #15: positive

📊 Consistency: 100% | Unique answers: 1 out of 15
Most common answer: positive


### Step 10: Add Chain-of-Thought to Data Extraction
Version 3 asks the model to reason step by step, one line per field, and gives every field a **fixed set of allowed values** (date as MM-DD, "not mentioned" when missing).

In [26]:
# 10.1 Version 3: step-by-step reasoning with fixed allowed values, then the JSON on the last line
# {feedback} is a placeholder; .replace() is used because the JSON braces would break an f-string

extraction_prompt_v3 = """
Extract the order information from this customer feedback:
"{feedback}"

Think step by step and write one short line for each step:
Step 1: Order number: digits only, or "not mentioned".
Step 2: Order date: MM-DD, or "not mentioned".
Step 3: Delivery speed: "fast", "slow" or "not mentioned".
Step 4: Packaging condition: "good", "damaged" or "not mentioned".
Step 5: Main issue: "damaged_packaging", "late_delivery", "wrong_item", "missing_order" or "none".

Then write the result on the last line in exactly this format:
FINAL JSON: {"order_number": "...", "order_date": "...", "delivery_speed": "...", "packaging_condition": "...", "issue_type": "..."}
"""

extraction_prompt_v3_test = extraction_prompt_v3.replace(
    "{feedback}", "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.")
display_response("Extraction prompt v3 (Chain-of-Thought)", extraction_prompt_v3_test,
                 {"system_message": extraction_system,
                  "approx_tokens": count_tokens_approx(extraction_prompt_v3_test)})


📝 Extraction prompt v3 (Chain-of-Thought)
Parameters: {'system_message': "The assistant extracts order data from TechFlow Solutions' customer feedback.", 'approx_tokens': 184}

Extract the order information from this customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

Think step by step and write one short line for each step:
Step 1: Order number: digits only, or "not mentioned".
Step 2: Order date: MM-DD, or "not mentioned".
Step 3: Delivery speed: "fast", "slow" or "not mentioned".
Step 4: Packaging condition: "good", "damaged" or "not mentioned".
Step 5: Main issue: "damaged_packaging", "late_delivery", "wrong_item", "missing_order" or "none".

Then write the result on the last line in exactly this format:
FINAL JSON: {"order_number": "...", "order_date": "...", "delivery_speed": "...", "packaging_condition": "...", "issue_type": "..."}




In [27]:
# 10.2 Compare only the final JSON: the reasoning lines are allowed to differ between runs

def cot_extraction_key(text):
    if "FINAL JSON:" in text:
        text = text.split("FINAL JSON:")[-1]
    return extraction_key(text)

print("✅ CoT comparison rule ready")

✅ CoT comparison rule ready


In [28]:
# 10.3 Test the Chain-of-Thought prompt fifteen times

extraction_v3_15 = test_prompt("Extraction", "v3", extraction_prompt_v3_test, cot_extraction_key, 15,
                               extraction_system)


📝 Extraction | v3 | 15 runs
🔄 Run #1: Step 1: Order number: 12345 Step 2: Order date: 03-15 Step 3: Delivery speed: fast Step 4: Packaging condition: damaged Step 5: Main issue: damaged_pa
🔄 Run #2: Step 1: Order number: 12345 Step 2: Order date: 03-15 Step 3: Delivery speed: fast Step 4: Packaging condition: damaged Step 5: Main issue: damaged_pa
🔄 Run #3: Step 1: Order number: 12345 Step 2: Order date: 03-15 Step 3: Delivery speed: fast Step 4: Packaging condition: damaged Step 5: Main issue: damaged_pa
🔄 Run #4: Step 1: Order number: 12345 Step 2: Order date: 03-15 Step 3: Delivery speed: fast Step 4: Packaging condition: damaged Step 5: Main issue: damaged_pa
🔄 Run #5: Step 1: Order number: 12345 Step 2: Order date: 03-15 Step 3: Delivery speed: fast Step 4: Packaging condition: damaged Step 5: Main issue: damaged_pa
🔄 Run #6: Step 1: Order number: 12345 Step 2: Order date: 03-15 Step 3: Delivery speed: fast Step 4: Packaging condition: damaged Step 5: Main issue: damaged_pa
🔄 Run

### Step 11: Add Few-Shot Examples and Structure to Product Description
Version 3 combines the v2 structure with two examples written in it. The examples use other products (USB-C hub, laptop stand), so the model copies the format.

In [29]:
# 11.1 Version 3: the v2 structure plus two examples written in it
# {product} and {price} are placeholders, so Step 12 can reuse the prompt with new products

product_prompt_v3 = """
Write a product description in exactly the same format as the examples.
Style: friendly and professional. No emojis, no headings, no bold text.

Structure:
Title: <product name and one key benefit, max 8 words>
<one paragraph of 40-60 words that mentions the price once>
- <feature 1>
- <feature 2>
- <feature 3>

Product: USB-C hub, $39.99
Description:
Title: USB-C Hub That Connects Everything at Once
Turn one USB-C port into a full workstation. This compact hub adds HDMI, USB-A and SD card slots, so you can connect a monitor, keyboard and camera at the same time. At $39.99, it is a simple upgrade for any laptop setup, at home or on the road.
- 4K HDMI output
- Three USB-A ports
- SD and microSD card reader

Product: Laptop stand, $24.99
Description:
Title: Laptop Stand for a Healthier Workday
Lift your screen to eye level and give your neck a break. This aluminium stand folds flat, fits in any bag and holds laptops up to 17 inches. For $24.99, it turns any table into a more comfortable workspace in seconds.
- Six adjustable heights
- Lightweight aluminium build
- Folds flat for travel

Product: {product}, {price}
Description:"""

product_prompt_v3_test = product_prompt_v3.replace("{product}", "Wireless mouse").replace("{price}", "$29.99")
display_response("Product prompt v3 (few-shot + structure)", product_prompt_v3_test,
                 {"system_message": product_system,
                  "approx_tokens": count_tokens_approx(product_prompt_v3_test)})


📝 Product prompt v3 (few-shot + structure)
Parameters: {'system_message': "The assistant is a copywriter for TechFlow Solutions' online electronics shop.", 'approx_tokens': 287}

Write a product description in exactly the same format as the examples.
Style: friendly and professional. No emojis, no headings, no bold text.

Structure:
Title: <product name and one key benefit, max 8 words>
<one paragraph of 40-60 words that mentions the price once>
- <feature 1>
- <feature 2>
- <feature 3>

Product: USB-C hub, $39.99
Description:
Title: USB-C Hub That Connects Everything at Once
Turn one USB-C port into a full workstation. This compact hub adds HDMI, USB-A and SD card slots, so you can connect a monitor, keyboard and camera at the same time. At $39.99, it is a simple upgrade for any laptop setup, at home or on the road.
- 4K HDMI output
- Three USB-A ports
- SD and microSD card reader

Product: Laptop stand, $24.99
Description:
Title: Laptop Stand for a Healthier Workday
Lift your screen

In [30]:
# 11.2 Test the combined approach fifteen times

product_v3_15 = test_prompt("Product", "v3", product_prompt_v3_test, product_key, 15, product_system)


📝 Product | v3 | 15 runs
🔄 Run #1: Title: Wireless Mouse for Effortless Navigation Experience seamless control with this wireless mouse designed for comfort and precision. At $29.99, it
🔄 Run #2: Title: Wireless Mouse for Effortless Navigation Experience smooth and precise control with this wireless mouse, designed for comfort and efficiency. P
🔄 Run #3: Title: Wireless Mouse for Effortless Navigation Experience smooth and precise control with this wireless mouse designed for comfort and efficiency. At
🔄 Run #4: Title: Wireless Mouse for Effortless Navigation Experience seamless control with our wireless mouse, designed for comfort and precision. Priced at $29
🔄 Run #5: Title: Wireless Mouse for Effortless Navigation Experience seamless control with our wireless mouse, designed for both comfort and precision. Priced a
🔄 Run #6: Title: Wireless Mouse for Effortless Navigation Experience seamless browsing with our wireless mouse designed for comfort and precision. Priced at $29
🔄 Run #7

In [31]:
# 11.3 Checkpoint: all version 3 results vs version 1 and 2 (15 runs)

tests_15 = pd.DataFrame(all_results).query("Runs == 15")
tests_15.pivot_table(index="Task", columns="Version", values="Consistency %", aggfunc="last")

Version,v1,v2,v3
Task,,,
Extraction,47,87,100
Product,40,100,100
Sentiment,40,100,100


## Part 5: Tuning for Different Tasks

### Step 12: Create Task Variations
The v3 prompts are tested on new inputs, 5 runs each: sarcasm, a plain fact and a mixed message for sentiment; two new products; feedback with a late delivery and with a missing order number.

In [32]:
# 12.1 New inputs the prompts have not seen

sentiment_variations = [
    ("Great, another broken charger. Just what I needed.", "negative"),   # sarcasm
    ("My order number is 4521.", "neutral"),                              # plain fact
    ("The delivery was late, but support fixed it quickly.", "mixed"),    # no single right answer
]

product_variations = [
    ("Mechanical keyboard", "$89.99"),
    ("Stainless steel water bottle", "$19.99"),
]

extraction_variations = [
    "Order #98765 arrived on April 2nd, two weeks late, but everything was intact.",
    "I never received my order from last week, and my receipt has no order number.",
]

print("✅ Variations ready")

✅ Variations ready


In [33]:
# 12.2 Best sentiment prompt (v3) on the variations

for message, expected in sentiment_variations:
    prompt = sentiment_prompt_v3.replace("{message}", message)
    test_prompt("Sentiment variation", "v3", prompt, sentiment_key, 5, sentiment_system)
    print(f"Expected: {expected}")


📝 Sentiment variation | v3 | 5 runs
🔄 Run #1: negative
🔄 Run #2: negative
🔄 Run #3: negative
🔄 Run #4: negative
🔄 Run #5: negative

📊 Consistency: 100% | Unique answers: 1 out of 5
Most common answer: negative
Expected: negative

📝 Sentiment variation | v3 | 5 runs
🔄 Run #1: neutral
🔄 Run #2: neutral
🔄 Run #3: neutral
🔄 Run #4: neutral
🔄 Run #5: neutral

📊 Consistency: 100% | Unique answers: 1 out of 5
Most common answer: neutral
Expected: neutral

📝 Sentiment variation | v3 | 5 runs
🔄 Run #1: positive
🔄 Run #2: positive
🔄 Run #3: positive
🔄 Run #4: positive
🔄 Run #5: positive

📊 Consistency: 100% | Unique answers: 1 out of 5
Most common answer: positive
Expected: mixed


In [34]:
# 12.3 Best product prompt (v3) on the variations

for product, price in product_variations:
    prompt = product_prompt_v3.replace("{product}", product).replace("{price}", price)
    test_prompt("Product variation", "v3", prompt, product_key, 5, product_system)


📝 Product variation | v3 | 5 runs
🔄 Run #1: Title: Mechanical Keyboard for Enhanced Typing Experience Transform your typing with this mechanical keyboard designed for comfort and precision. With
🔄 Run #2: Title: Mechanical Keyboard for Enhanced Typing Experience Elevate your typing experience with this high-performance mechanical keyboard. Designed for 
🔄 Run #3: Title: Mechanical Keyboard for Ultimate Typing Precision Experience the joy of typing with our mechanical keyboard, designed for both gamers and profe
🔄 Run #4: Title: Mechanical Keyboard for Ultimate Typing Experience Elevate your typing game with this premium mechanical keyboard designed for both gamers and 
🔄 Run #5: Title: Mechanical Keyboard for Ultimate Typing Precision Experience the satisfying click and tactile feedback of our mechanical keyboard, designed for

📊 Consistency: 100% | Unique answers: 1 out of 5
Most common answer: 3 list lines | medium (50-120 words)

📝 Product variation | v3 | 5 runs
🔄 Run #1: Title: S

In [35]:
# 12.4 Best extraction prompt (v3) on the variations

for feedback in extraction_variations:
    prompt = extraction_prompt_v3.replace("{feedback}", feedback)
    test_prompt("Extraction variation", "v3", prompt, cot_extraction_key, 5, extraction_system)


📝 Extraction variation | v3 | 5 runs
🔄 Run #1: Step 1: Order number: 98765 Step 2: Order date: 04-02 Step 3: Delivery speed: slow Step 4: Packaging condition: good Step 5: Main issue: late_delivery
🔄 Run #2: Step 1: Order number: 98765 Step 2: Order date: 04-02 Step 3: Delivery speed: slow Step 4: Packaging condition: good Step 5: Main issue: late_delivery
🔄 Run #3: Step 1: Order number: 98765 Step 2: Order date: 04-02 Step 3: Delivery speed: slow Step 4: Packaging condition: good Step 5: Main issue: late_delivery
🔄 Run #4: Step 1: Order number: 98765 Step 2: Order date: 04-02 Step 3: Delivery speed: slow Step 4: Packaging condition: good Step 5: Main issue: late_delivery
🔄 Run #5: Step 1: Order number: 98765 Step 2: Order date: not mentioned Step 3: Delivery speed: slow Step 4: Packaging condition: good Step 5: Main issue: late_

📊 Consistency: 80% | Unique answers: 2 out of 5
Most common answer: {"delivery_speed": "slow", "issue_type": "late_delivery", "order_date": "04-02", "order_

### Step 12: Variation results

**Sentiment (v3 prompt, 5 runs each)**
- **Sarcasm:** "Great, another broken charger. Just what I needed." → negative in 5 of 5 runs (expected: negative). 
Works.
- **Plain fact:** "My order number is 4521." → neutral in 5 of 5 runs (expected: neutral). 
Works.
- **Mixed message:** "The delivery was late, but support fixed it quickly." → positive in 5 of 5 runs (expected: mixed). 
Consistent, but there is no right answer: the label set has no "mixed" option, so the model picks one side.

**Product description (v3 prompt, 5 runs each)**
- **Mechanical keyboard, $89.99:** title, paragraph and 3 bullets in 5 of 5 runs. 
Works; only the title wording varied ("Enhanced Typing Experience", "Ultimate Typing Precision").
- **Stainless steel water bottle, $19.99:** title, paragraph and 3 bullets in 5 of 5 runs, with the same title every time. 
Works.

**Data extraction (v3 prompt, 5 runs each)**
- **Late delivery:** "Order #98765 arrived on April 2nd, two weeks late, but everything was intact." → 98765, 04-02, slow, good, late_delivery in 4 of 5 runs; 1 run gave the order date as "not mentioned" (80%). 
Partly works: April 2nd is the arrival date, not the order date, so the four consistent runs put it in the wrong field and the outlier was the correct reading.
- **Missing order:** "I never received my order from last week, and my receipt has no order number." → every field "not mentioned" and issue missing_order in 5 of 5 runs. 
Works.

**Adjustments needed**
- **Sentiment:** add a "mixed" label, or agree a rule with TechFlow (for example, label the customer's final state).
- **Product description:** change "electronics shop" in the role to a neutral "online shop", because the catalogue includes products like the water bottle.
- **Data extraction:** add a rule ("an arrival or delivery date is not an order date") or a separate delivery_date field.

**What this shows:** The two weak spots are about content, not format: a label set without "mixed", and a date field the model can confuse. Consistency is not the same as correctness: the late-delivery case was 80% consistent, and the consistent answer was the wrong one.

### Step 13: Final Evaluation and Comparison
The final v3 prompts run 15 times each, then the comparison report: consistency per version, improvement from v1 to v3, and whether the >80% target is met.

In [36]:
# 13.1 Run the final prompts (version 3) fifteen times each

sentiment_final = test_prompt("Sentiment", "v3 final", sentiment_prompt_v3_test, sentiment_key, 15,
                              sentiment_system)
product_final = test_prompt("Product", "v3 final", product_prompt_v3_test, product_key, 15,
                            product_system)
extraction_final = test_prompt("Extraction", "v3 final", extraction_prompt_v3_test, cot_extraction_key, 15,
                               extraction_system)


📝 Sentiment | v3 final | 15 runs
🔄 Run #1: positive
🔄 Run #2: positive
🔄 Run #3: positive
🔄 Run #4: positive
🔄 Run #5: positive
🔄 Run #6: positive
🔄 Run #7: positive
🔄 Run #8: positive
🔄 Run #9: positive
🔄 Run #10: positive
🔄 Run #11: positive
🔄 Run #12: positive
🔄 Run #13: positive
🔄 Run #14: positive
🔄 Run #15: positive

📊 Consistency: 100% | Unique answers: 1 out of 15
Most common answer: positive

📝 Product | v3 final | 15 runs
🔄 Run #1: Title: Wireless Mouse for Effortless Navigation Experience seamless control with our wireless mouse designed for comfort and precision. Priced at $29.
🔄 Run #2: Title: Wireless Mouse for Effortless Navigation Experience seamless control and comfort with our wireless mouse. Designed for precision and ease, this
🔄 Run #3: Title: Wireless Mouse for Effortless Navigation Experience smooth and responsive control with our wireless mouse, designed for comfort and precision. 
🔄 Run #4: Title: Wireless Mouse for Effortless Navigation Experience seamless co

In [37]:
# 13.2 Comparison report and improvement metrics (15 runs)

tests_15 = pd.DataFrame(all_results).query("Runs == 15")
report = tests_15.pivot_table(index="Task", columns="Version", values="Consistency %", aggfunc="last")
report = report.reindex(["Sentiment", "Product", "Extraction"])  # same task order as the brief
report["Improvement (points)"] = report["v3 final"] - report["v1"]
report["Target >80% met"] = report["v3 final"] > 80
report

Version,v1,v2,v3,v3 final,Improvement (points),Target >80% met
Task,,,,,,
Sentiment,40,100,100,100,60,True
Product,40,100,100,100,60,True
Extraction,47,87,100,100,53,True


In [38]:
# 13.3 Final version 3 prompts as sent in 13.1, with their approximate size compared to version 1

final_prompts = [("Sentiment", sentiment_system, sentiment_prompt_v1, sentiment_prompt_v3_test),
                 ("Product description", product_system, product_prompt_v1, product_prompt_v3_test),
                 ("Data extraction", extraction_system, extraction_prompt_v1, extraction_prompt_v3_test)]

for task, system_message, prompt_v1, prompt_v3 in final_prompts:
    display_response(f"{task} (v3)", prompt_v3,
                     {"system_message": system_message,
                      "approx_tokens_v1": count_tokens_approx(prompt_v1),
                      "approx_tokens_v3": count_tokens_approx(prompt_v3)})


📝 Sentiment (v3)
Parameters: {'system_message': "The assistant is a sentiment classifier for TechFlow Solutions' customer service team.", 'approx_tokens_v1': 21, 'approx_tokens_v3': 122}

Classify the sentiment of the customer message as positive, negative or neutral.
Answer with exactly one lowercase word, like in the examples.

Customer message: "The app crashes every time I open it. Very disappointed."
Sentiment: negative

Customer message: "My package arrived on Tuesday."
Sentiment: neutral

Customer message: "Great service, the team solved my problem in minutes!"
Sentiment: positive

Customer message: "I love this product! It's exactly what I needed."
Sentiment:


📝 Product description (v3)
Parameters: {'system_message': "The assistant is a copywriter for TechFlow Solutions' online electronics shop.", 'approx_tokens_v1': 17, 'approx_tokens_v3': 287}

Write a product description in exactly the same format as the examples.
Style: friendly and professional. No emojis, no headings, n

## Final Report

### Consistency across versions (15 runs, from cell 13.2)

| Task | v1 | v2 | v3 | v3 final | Improvement (v1 → v3) | Target >80% |
|---|---|---|---|---|---|---|
| Sentiment | 40% | 100% | 100% | 100% | +60 points | Met |
| Product description | 40% | 100% | 100% | 100% | +60 points | Met |
| Data extraction | 47% | 87% | 100% | 100% | +53 points | Met |

All three final prompts reached 100% over 15 runs, well above the 80% target, and held it when the final prompts were run again. Failure patterns are documented in Step 5 (v1 prompts) and Step 12 (new inputs).

### What the numbers do not show
- **v2 already fixed sentiment and product format.** A role, a closed label set and a one-word rule took sentiment from 40% to 100%; a structure template took product descriptions from 40% to 100%.
- **v2 extraction failed on content, not syntax.** All 15 answers were valid JSON (the parsing issues the brief warned about did not occur), but the date came back as "March 15th" 13 times and "2023-03-15" twice. The year 2023 is not in the feedback: the model invented it. The fixed MM-DD format in v3 removed both the variation and the invented year.
- **The examples also fixed content.** In v2 the model invented a brand name in 9 of 15 runs (SwiftClick, SwiftConnect, SmoothGlide, SmartConnect). With two examples in v3, every title used the product name it was given ("Wireless Mouse for Effortless Navigation" in 15 of 15 runs).
- **Consistency is not correctness.** On new inputs (Step 12) the prompts stayed consistent, but a mixed message has no correct label, and an arrival date was read as the order date in 4 of 5 runs.

### Why each improvement was made
- **Sentiment:** v2 added a role and a closed label set, so the model stops explaining, and a one-word rule at the end (recency bias), so it stops writing sentences. v3 added one example per label, so the answer format is shown, not only described; it also read sarcasm correctly on new inputs.
- **Product description:** v2 added a structure template with length limits, so every answer has the same parts and size. v3 added two examples written in that template, so the model sees what following it looks like and stays with the product name it is given.
- **Data extraction:** v2 asked for JSON only with named fields, so the structure is fixed. v3 added step-by-step reasoning with allowed values for every field, so the content is fixed too.


### What worked best per task type
- **Classification:** the closed label set did most of the work (100% already in v2); few-shot examples kept it right on sarcasm and plain facts.
- **Generation:** the structure template fixed the format, and the examples anchored the content (product name, tone). The paragraph wording still varies, which is wanted.
- **Extraction:** Chain-of-Thought with a fixed set of allowed values per field, the only version with a stable date format and no invented year.
- **Trade-off:** the v3 prompts are about 5 to 17 times longer than v1 in approximate tokens (sentiment 21 → 122, product 17 → 287, extraction 35 → 184; see cell 13.3), so the consistency gain costs input tokens on every call. At production volume, trim wording and examples (token optimization, course notebook).

### Next steps for production
- Add a "mixed" sentiment label (or an agreed rule) and a separate delivery_date field, then re-test the Step 12 inputs.
- Use a neutral "online shop" role if the catalogue goes beyond electronics.
- Run classification and extraction at temperature=0 or with a fixed seed, where consistency matters most; keep 0.7 for product copy.

## Bonus: Challenge 4, Evaluation Framework

`evaluate_prompt` runs a prompt several times and scores it on three checks. Each score is the share of runs that pass the check; the overall score is the average of the three.

| Task | Format adherence | Content quality | Consistency |
|---|---|---|---|
| Sentiment | exactly one allowed label | the expected sentiment | same label every run |
| Product description | title, 3 bullets, 50-120 words, no bold | names the product and the price | same format every run |
| Data extraction | valid JSON | all required fields filled | same data every run |

The v1 and v3 prompts of all three tasks are evaluated with 10 runs each, using the same comparison rules as Steps 3 to 13.

In [39]:
# B.1 Evaluation framework: one function that scores a prompt on format, content and consistency

def evaluate_prompt(name, prompt, format_check, content_check, key_function, n_runs=10,
                    system_message="You are a helpful assistant."):
    responses = run_multiple(prompt, n_runs, system_message)

    # Format adherence: share of runs with the required format (e.g. valid JSON, correct length)
    format_score = sum(format_check(r) for r in responses) / n_runs * 100

    # Content quality: share of runs that contain the required content (e.g. required fields)
    content_score = sum(content_check(r) for r in responses) / n_runs * 100

    # Consistency: share of runs that give the most common answer (same rule as Steps 3 to 13)
    keys = [key_function(r) for r in responses]
    consistency_score = Counter(keys).most_common(1)[0][1] / n_runs * 100

    print(f"✅ {name}: {n_runs} runs evaluated")
    return {"Prompt": name,
            "Format %": round(format_score),
            "Content %": round(content_score),
            "Consistency %": round(consistency_score),
            "Overall %": round((format_score + content_score + consistency_score) / 3)}

print("✅ Evaluation function ready")

✅ Evaluation function ready


In [40]:
# B.2 Checks for each task: what "right format" and "right content" mean

# Sentiment: format = exactly one allowed label; content = the expected sentiment of the test message
def sentiment_format(text):
    return text.strip() in ["positive", "negative", "neutral"]

def sentiment_content(text):
    return "positive" in text.lower()


# Product description: format = title line, 3 bullets, 50-120 words, no bold; content = product and price
def product_format(text):
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    bullets = sum(line.startswith("- ") for line in lines)
    words = len(text.split())
    return (len(lines) > 0 and lines[0].startswith("Title:") and bullets == 3
            and 50 <= words <= 120 and "**" not in text)

def product_content(text):
    return "mouse" in text.lower() and "$29.99" in text


# Data extraction: format = valid JSON (after "FINAL JSON:" for the CoT prompt); content = required fields filled
REQUIRED_FIELDS = ["order_number", "order_date", "delivery_speed", "packaging_condition"]

def parse_json_answer(text):
    if "FINAL JSON:" in text:
        text = text.split("FINAL JSON:")[-1]
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        data = json.loads(cleaned)
        return data if isinstance(data, dict) else None
    except json.JSONDecodeError:
        return None

def extraction_format(text):
    return parse_json_answer(text) is not None

def extraction_content(text):
    data = parse_json_answer(text)
    return data is not None and all(data.get(field) for field in REQUIRED_FIELDS)

print("✅ Task checks ready")

✅ Task checks ready


In [41]:
# B.3 Evaluate the v1 and v3 prompts of all three tasks (10 runs each, about 60 API calls)

evaluations = [
    evaluate_prompt("Sentiment v1", sentiment_prompt_v1, sentiment_format, sentiment_content, sentiment_key),
    evaluate_prompt("Sentiment v3", sentiment_prompt_v3_test, sentiment_format, sentiment_content, sentiment_key,
                    system_message=sentiment_system),
    evaluate_prompt("Product v1", product_prompt_v1, product_format, product_content, product_key),
    evaluate_prompt("Product v3", product_prompt_v3_test, product_format, product_content, product_key,
                    system_message=product_system),
    evaluate_prompt("Extraction v1", extraction_prompt_v1, extraction_format, extraction_content, extraction_key),
    evaluate_prompt("Extraction v3", extraction_prompt_v3_test, extraction_format, extraction_content,
                    cot_extraction_key, system_message=extraction_system),
]

scorecard = pd.DataFrame(evaluations)
scorecard

✅ Sentiment v1: 10 runs evaluated
✅ Sentiment v3: 10 runs evaluated
✅ Product v1: 10 runs evaluated
✅ Product v3: 10 runs evaluated
✅ Extraction v1: 10 runs evaluated
✅ Extraction v3: 10 runs evaluated


,Prompt,Format %,Content %,Consistency %,Overall %
0,Sentiment v1,0,100,30,43
1,Sentiment v3,100,100,100,100
2,Product v1,0,100,50,50
3,Product v3,100,100,100,100
4,Extraction v1,0,0,40,13
5,Extraction v3,100,100,100,100


### Bonus results

| Prompt | Format % | Content % | Consistency % | Overall % |
|---|---|---|---|---|
| Sentiment v1 | 0 | 100 | 30 | 43 |
| Sentiment v3 | 100 | 100 | 100 | 100 |
| Product v1 | 0 | 100 | 50 | 50 |
| Product v3 | 100 | 100 | 100 | 100 |
| Extraction v1 | 0 | 0 | 40 | 13 |
| Extraction v3 | 100 | 100 | 100 | 100 |

**What the scorecard shows**

- **Format is where every v1 prompt failed:** 0% in all three tasks. No v1 answer was a single label, a title plus 3 bullets within 50-120 words, or valid JSON.
- **The v1 content was often right:** sentiment v1 named the correct sentiment and product v1 named the product and price in 100% of runs. The model knew the answer; the prompt never said how to deliver it. Consistency alone (30% and 50%) hides this, which is why the three checks are scored separately.
- **Extraction v1 scored lowest (13% overall):** without

## EXTRA Analysis That I added: Cost comparison across versions

How the cost per call is estimated:

- **Input tokens:** system message plus prompt, measured with `count_tokens_approx` (about 4 characters per token).
- **Output tokens:** the average answer size in each version's 15-run test.
- **Prices:** gpt-4o-mini at $0.15 per 1M input tokens and $0.60 per 1M output tokens, the course notebook's example figures; check the current price list before quoting them to a client.
- **Monthly cost:** the course notebook's scenario of 1,000 calls per day for 30 days.

No new API calls: the tables are built from the prompts and the answers already recorded in Steps 5 to 11.

In [42]:
# C.1 Pricing and cost helper (from the course notebook, section 5: check current pricing)

INPUT_COST_PER_1K = 0.00015   # $0.15 per 1M input tokens (gpt-4o-mini)
OUTPUT_COST_PER_1K = 0.0006   # $0.60 per 1M output tokens (gpt-4o-mini)

def calculate_cost(input_tokens, output_tokens):
    """Calculate cost based on token usage"""
    input_cost = (input_tokens / 1000) * INPUT_COST_PER_1K
    output_cost = (output_tokens / 1000) * OUTPUT_COST_PER_1K
    return input_cost + output_cost

print("✅ Cost helper ready")

✅ Cost helper ready


In [43]:
# C.2 Cost of every version: prompt size (input) and average answer size (output) from the 15-run tests

versions = [
    ("Sentiment", "v1", "You are a helpful assistant.", sentiment_prompt_v1, sentiment_v1_15),
    ("Sentiment", "v2", sentiment_system, sentiment_prompt_v2, sentiment_v2_15),
    ("Sentiment", "v3", sentiment_system, sentiment_prompt_v3_test, sentiment_v3_15),
    ("Product", "v1", "You are a helpful assistant.", product_prompt_v1, product_v1_15),
    ("Product", "v2", product_system, product_prompt_v2, product_v2_15),
    ("Product", "v3", product_system, product_prompt_v3_test, product_v3_15),
    ("Extraction", "v1", "You are a helpful assistant.", extraction_prompt_v1, extraction_v1_15),
    ("Extraction", "v2", extraction_system, extraction_prompt_v2, extraction_v2_15),
    ("Extraction", "v3", extraction_system, extraction_prompt_v3_test, extraction_v3_15),
]

cost_rows = []
for task, version, system_message, prompt, responses in versions:
    input_tokens = count_tokens_approx(system_message + prompt)
    output_tokens = sum(count_tokens_approx(r) for r in responses) / len(responses)  # average answer size
    cost_per_call = calculate_cost(input_tokens, output_tokens)
    cost_rows.append({"Task": task, "Version": version,
                      "Input tokens": input_tokens,
                      "Output tokens": round(output_tokens),
                      "Cost per call ($)": round(cost_per_call, 6),
                      "Monthly cost ($)": round(cost_per_call * 1000 * 30, 2)})  # 1,000 calls/day, 30 days

cost_table = pd.DataFrame(cost_rows)
cost_table

,Task,Version,Input tokens,Output tokens,Cost per call ($),Monthly cost ($)
0,Sentiment,v1,28,22,0.000018,0.53
1,Sentiment,v2,77,2,0.000013,0.38
2,Sentiment,v3,143,2,0.000023,0.68
3,Product,v1,24,491,0.000298,8.94
4,Product,v2,101,122,0.000088,2.65
5,Product,v3,306,101,0.000106,3.19
6,Extraction,v1,42,45,0.000033,0.99
7,Extraction,v2,100,30,0.000033,0.99
8,Extraction,v3,203,81,0.000079,2.37


In [44]:
# C.3 Compare the versions: monthly cost per task, and v3 relative to v1

monthly = cost_table.pivot_table(index="Task", columns="Version", values="Monthly cost ($)", aggfunc="last")
monthly = monthly.reindex(["Sentiment", "Product", "Extraction"])  # same task order as the brief
monthly["v3 vs v1"] = (monthly["v3"] / monthly["v1"]).round(2).astype(str) + "x"
monthly

Version,v1,v2,v3,v3 vs v1
Task,,,,
Sentiment,0.53,0.38,0.68,1.28x
Product,8.94,2.65,3.19,0.36x
Extraction,0.99,0.99,2.37,2.39x


### Cost results

| Task | v1 ($/month) | v2 ($/month) | v3 ($/month) | v3 vs v1 |
|---|---|---|---|---|
| Sentiment | 0.53 | 0.38 | 0.68 | 1.28x |
| Product description | 8.94 | 2.65 | 3.19 | 0.36x |
| Data extraction | 0.99 | 0.99 | 2.37 | 2.39x |
| All three tasks | 10.46 | 4.02 | 6.24 | 0.60x |

**What the costs show**

- **Longer prompts did not mean higher cost.** The v3 prompts are 5 to 13 times longer in input tokens, yet the three v3 prompts together cost about 40% less per month than v1 ($6.24 against $10.46).
- **Output tokens drive the cost.** They are priced four times higher than input tokens, and v1 product descriptions averaged 491 output tokens: about 99% of that prompt's cost. The v3 template cut answers to 101 tokens, making product v3 64% cheaper than v1 despite a 13 times longer prompt.
- **Sentiment:** the answer shrank from 22 tokens to 2, but the few-shot examples add 115 input tokens, so v3 costs 1.28x v1. v2 is the cheapest sentiment prompt and already scored 100%.
- **Extraction is the one real cost increase (2.39x).** Chain-of-Thought adds input (203 tokens) and output, because the reasoning lines are paid for on every call (81 tokens against 30 for v2's plain JSON).
- **The amounts are small in absolute terms:** no version passes $9 a month at 1,000 calls per day. At this scale consistency decides the choice, not cost.

**Optimisation options:** v2 is the cheapest set ($4.02) but its extraction only reached 87% consistency (meeting expectations). A cheaper extraction prompt would keep the fixed allowed values but drop the written reasoning; that would need its own 15-run test before replacing v3.